In [1]:

# 0. INSTALL DEPENDENCIES
%pip install flask flask-cors pyngrok imbalanced-learn scikit-learn pandas numpy matplotlib seaborn scipy -q


[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:

import os, sys, time, warnings, csv, threading, pickle, io, base64
from datetime import datetime
from math import gamma as G

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from sklearn.svm             import SVC
from sklearn.preprocessing   import StandardScaler
from sklearn.impute          import SimpleImputer
from sklearn.pipeline        import Pipeline
from sklearn.model_selection import (StratifiedKFold, cross_val_score,
                                     train_test_split)
from sklearn.metrics         import (accuracy_score, classification_report,
                                     confusion_matrix, ConfusionMatrixDisplay,
                                     roc_auc_score, roc_curve,
                                     precision_recall_curve,
                                     average_precision_score,
                                     f1_score, matthews_corrcoef)

from flask import Flask, request, jsonify, render_template_string, send_file
from flask_cors import CORS

warnings.filterwarnings('ignore')
SEED = 42
np.random.seed(2024)
print('Imports OK')


Imports OK


## Section 1 — Data Loading & Preprocessing

In [3]:
# 1. DATA LOADING & PREPROCESSING

COLS = ['age','sex','cp','trestbps','chol','fbs','restecg',
        'thalach','exang','oldpeak','slope','ca','thal','target']

DATA_FILES = [
    os.path.expanduser("~/Desktop/Toni's code/Đồ án chuyên ngành/processed.cleveland.data"),
    os.path.expanduser("~/Desktop/Toni's code/Đồ án chuyên ngành/processed.hungarian.data"),
    os.path.expanduser("~/Desktop/Toni's code/Đồ án chuyên ngành/processed.va.data"),
    os.path.expanduser("~/Desktop/Toni's code/Đồ án chuyên ngành/processed.switzerland.data"),
]

frames = []
for f in DATA_FILES:
    df = pd.read_csv(f, header=None, names=COLS, na_values='?')
    frames.append(df)

data = pd.concat(frames, ignore_index=True)
print(f"[DATA] Raw combined shape: {data.shape}")
print(f"  Missing trước xử lý: {data.isnull().sum().sum()} giá trị")

# Nhị phân hoá target: 0 = không bệnh, 1 = có bệnh
data['target'] = (data['target'] > 0).astype(int)

# --- XỬ LÝ MISSING VALUES BẰNG KNN IMPUTER ---
# Dùng KNNImputer (k=5, weights='distance') thay vì loại bỏ cột
# để giữ nguyên đủ 13 đặc trưng gốc của UCI Heart Disease.
# weights='distance': mẫu gần hơn có ảnh hưởng nhiều hơn khi điền.
from sklearn.impute import KNNImputer
imputer = KNNImputer(n_neighbors=5, weights='distance')
data_arr = imputer.fit_transform(data)
data = pd.DataFrame(data_arr, columns=COLS)

# Làm tròn về giá trị hợp lệ cho từng cột (vectorized — không dùng apply)
# ca: số mạch máu chính bị hẹp, chỉ nhận 0,1,2,3
data['ca']      = data['ca'].round(0).clip(0, 3).astype(int)
# thal: chỉ nhận 3 (normal), 6 (fixed defect), 7 (reversible defect)
_thal = data['thal'].values
data['thal']    = np.where(_thal <= 4.5, 3, np.where(_thal <= 6.5, 6, 7)).astype(int)
# Các cột nhị phân
for col in ['sex', 'fbs', 'exang']:
    data[col] = data[col].round(0).clip(0, 1).astype(int)
# Các cột ordinal/categorical khác
data['restecg'] = data['restecg'].round(0).clip(0, 2).astype(int)
data['slope']   = data['slope'].round(0).clip(1, 3).astype(int)
data['cp']      = data['cp'].round(0).clip(1, 4).astype(int)
data['age']     = data['age'].round(0).astype(int)
data['target']  = data['target'].round(0).astype(int)
# Cột liên tục: giữ 1 chữ số thập phân
for col in ['trestbps', 'chol', 'thalach', 'oldpeak']:
    data[col] = data[col].round(1)

print(f"  Missing sau KNN Imputer : {data.isnull().sum().sum()}")
print(f"  [DATA] Final shape      : {data.shape} — đủ 13 đặc trưng")

FEATURES = [c for c in data.columns if c != 'target']
X_raw = data[FEATURES].values.astype(np.float64)
y     = data['target'].values.astype(int)
n_feat = len(FEATURES)

print(f"  Samples  : {len(y)}  |  Disease={y.sum()}  Healthy={len(y)-y.sum()}")
print(f"  Features : {n_feat} → {FEATURES}")


[DATA] Raw combined shape: (920, 14)
  Missing trước xử lý: 1759 giá trị
  Missing sau KNN Imputer : 0
  [DATA] Final shape      : (920, 14) — đủ 13 đặc trưng
  Samples  : 920  |  Disease=509  Healthy=411
  Features : 13 → ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal']


## Section 1.5 — Missing Values & Class Balancing (SMOTE)

In [4]:
# 1.5 MISSING VALUES & CLASS IMBALANCE ANALYSIS
# ⚠  FIX: SMOTE được dời sang Section 4 (chỉ apply trên train split)
#    → Tránh data leakage: mẫu synthetic không rò rỉ vào test set
print("\n" + "═"*70)
print("  MISSING VALUES & CLASS IMBALANCE ANALYSIS")
print("═"*70)

print("\n  [MISSING] After KNN Imputer:")
missing_after = pd.DataFrame(X_raw, columns=FEATURES).isnull().sum()
print(f"  Total missing values: {missing_after.sum()}")

print("\n  [IMBALANCE] Target distribution (ORIGINAL — trước SMOTE):")
unique, counts = np.unique(y, return_counts=True)
for u, c in zip(unique, counts):
    label = 'Disease' if u == 1 else 'No Disease'
    print(f"    {label:<15}: {c:4d} ({c/len(y)*100:5.1f}%)")
imbalance_ratio = counts.max() / counts.min()
print(f"  Imbalance Ratio: {imbalance_ratio:.2f}:1")

print("\n  [NOTE] SMOTE sẽ được apply SAU train_test_split tại Section 4")
print("         → Đảm bảo không có data leakage vào test set")
# X_raw và y GIỮ NGUYÊN — D-DOA optimize trên data gốc (không SMOTE)
# Lý do: D-DOA dùng StratifiedKFold bên trong → class weight tự cân bằng
#         SMOTE trên toàn bộ data trước khi split gây inflate CV score



══════════════════════════════════════════════════════════════════════
  MISSING VALUES & CLASS IMBALANCE ANALYSIS
══════════════════════════════════════════════════════════════════════

  [MISSING] After KNN Imputer:
  Total missing values: 0

  [IMBALANCE] Target distribution (ORIGINAL — trước SMOTE):
    No Disease     :  411 ( 44.7%)
    Disease        :  509 ( 55.3%)
  Imbalance Ratio: 1.24:1

  [NOTE] SMOTE sẽ được apply SAU train_test_split tại Section 4
         → Đảm bảo không có data leakage vào test set


## Section 2 — Deep Drizzle Optimization Algorithm (Class Definition)

In [5]:
# ═══════════════════════════════════════════════════════════════
#  SECTION 2 — DEEP DRIZZLE OPTIMIZATION ALGORITHM (DeepDOA)
#  Phiên bản v3 — tối ưu tốc độ & accuracy ≥ 85%:
#    ✔ SEED định nghĩa rõ ràng (Cell 1)
#    ✔ Local RNG (np.random.default_rng) — không reset global seed
#    ✔ Cơ chế 1: PSO chuẩn có cả gbest VÀ personal_best
#    ✔ Cơ chế 2: Global attraction đúng: β_g*(gbest − pos)
#    ✔ Greedy acceptance: pop_pos chỉ update khi f cải thiện
#    ✔ Levy Surge: chỉ apply cho droplet bị stagnation
# ═══════════════════════════════════════════════════════════════

from math import gamma as gamma_func

class DeepDOA:
    """
    Deep Drizzle Optimization Algorithm — phiên bản hoàn chỉnh
    ────────────────────────────────────────────────────────────
    Tối ưu đồng thời:
      - Feature weights  w[i] ∈ [w_min, w_max]  (dim = n_features)
      - log10(C)        ∈ [logC_min, logC_max]
      - log10(gamma)    ∈ [logg_min, logg_max]

    5 cơ chế:
      1. Raindrop Stratification : PSO-style, có cả gbest & pbest
      2. Global Attraction       : kéo về gbest trực tiếp
      3. Neighbourhood Sharing   : học từ hàng xóm tốt nhất
      4. Condensation Perturbation: elite mean + Gaussian noise
      5. Levy Surge              : chỉ cho droplet bị stagnation
    """

    def __init__(
        self,
        n_droplets=30, n_iter=2000,
        w_min=0.001, w_max=5.0,
        logC_min=-1.0, logC_max=4.0,
        logg_min=-4.0, logg_max=1.0,
        alpha_max=0.90, alpha_min=0.30,
        bl_init=0.35, bl_end=0.70,
        bg_init=0.70, bg_end=0.20,
        levy_init=0.15, levy_min=0.002,
        cond_pct=0.25, cond_sigma=0.10,
        surge_every=10, surge_str=0.80,
        nb_init=5, nb_max=15,
        elite_k=5, cv_folds=5,
        stagnation_thresh=12,
        verbose=True,
    ):
        self.n_droplets        = n_droplets
        self.n_iter            = n_iter
        self.w_min             = w_min
        self.w_max             = w_max
        self.logC_min          = logC_min
        self.logC_max          = logC_max
        self.logg_min          = logg_min
        self.logg_max          = logg_max
        self.alpha_max         = alpha_max
        self.alpha_min         = alpha_min
        self.bl_init           = bl_init
        self.bl_end            = bl_end
        self.bg_init           = bg_init
        self.bg_end            = bg_end
        self.levy_init         = levy_init
        self.levy_min          = levy_min
        self.cond_pct          = cond_pct
        self.cond_sigma        = cond_sigma
        self.surge_every       = surge_every
        self.surge_str         = surge_str
        self.nb_init           = nb_init
        self.nb_max            = nb_max
        self.elite_k           = elite_k
        self.cv_folds          = cv_folds
        self.stagnation_thresh = stagnation_thresh
        self.verbose           = verbose

        # Kết quả sau khi chạy
        self.best_weights_  = None
        self.best_C_        = None
        self.best_gamma_    = None
        self.best_fitness_  = None
        self.n_evaluations_ = 0

        # Tracking history
        self.hist_best  = []
        self.hist_mean  = []
        self.hist_std   = []
        self.alpha_hist = []
        self.bl_hist    = []
        self.bg_hist    = []
        self.levy_hist  = []
        self.hist_C     = []
        self.hist_gam   = []

    # ── Levy flight (Mantegna algorithm) ─────────────────────
    @staticmethod
    def _levy(dim, beta=1.5, rng=None):
        if rng is None:
            rng = np.random.default_rng()
        num   = gamma_func(1 + beta) * np.sin(np.pi * beta / 2)
        denom = gamma_func((1 + beta) / 2) * beta * 2 ** ((beta - 1) / 2)
        sigma = (num / denom) ** (1 / beta)
        u = rng.normal(0, sigma, dim)
        v = rng.normal(0, 1,     dim)
        return u / (np.abs(v) ** (1 / beta))

    def _alpha(self, t):
        ratio = t / self.n_iter
        return self.alpha_max - (self.alpha_max - self.alpha_min) * ratio

    def _beta_local(self, t):
        ratio = t / self.n_iter
        return self.bl_init + (self.bl_end - self.bl_init) * ratio

    def _beta_global(self, t):
        ratio = t / self.n_iter
        return self.bg_init + (self.bg_end - self.bg_init) * ratio

    def _levy_strength(self, t):
        ratio = t / self.n_iter
        return max(self.levy_min, self.levy_init * (1 - ratio))

    def _nb_size(self, t):
        ratio = t / self.n_iter
        return int(self.nb_init + (self.nb_max - self.nb_init) * ratio)

    # ── Clip về bounds ────────────────────────────────────────
    def _clip(self, pos, n_feat):
        pos[:n_feat]  = np.clip(pos[:n_feat],  self.w_min,    self.w_max)
        pos[n_feat]   = np.clip(pos[n_feat],   self.logC_min, self.logC_max)
        pos[n_feat+1] = np.clip(pos[n_feat+1], self.logg_min, self.logg_max)
        return pos

    # ── Fitness function ──────────────────────────────────────
    # QUAN TRỌNG: dùng Pipeline(StandardScaler → SVM) trong cross_val_score
    # → scaler fit trên train-fold, transform test-fold → không data leakage
    # → C và gamma được tối ưu trên data đã scale → nhất quán với hold-out
    # → SVM-RBF hoạt động đúng thiết kế (cực kỳ nhạy với scale của input)
    def _fitness(self, pos, X, y, n_feat, cv):
        self.n_evaluations_ += 1
        w   = pos[:n_feat]
        C   = 10.0 ** pos[n_feat]
        gam = 10.0 ** pos[n_feat + 1]
        Xw  = X * w
        pipe = Pipeline([
            ('scaler', StandardScaler()),
            ('svm',    SVC(C=C, kernel='rbf', gamma=gam,
                           probability=False, random_state=SEED)),
        ])
        scores = cross_val_score(pipe, Xw, y, cv=cv, scoring='roc_auc', n_jobs=-1)
        return 1.0 - scores.mean()   # minimize → 0 là tốt nhất

    # ── Main optimization loop ────────────────────────────────
    def optimize(self, X, y):
        n_feat = X.shape[1]
        D      = n_feat + 2   # weights + logC + logg
        cv     = StratifiedKFold(n_splits=self.cv_folds, shuffle=True, random_state=SEED)

        # ── Local RNG — KHÔNG reset global numpy seed ─────────
        rng = np.random.default_rng(SEED)

        # ── Khởi tạo quần thể — Latin Hypercube + seeded good regions ──
        # 80% random uniform (đảm bảo đa dạng)
        # 20% seeded tại vùng C=[1,100], gamma=[0.001,0.1] — vùng tốt cho SVM-RBF tabular
        pop_pos = np.zeros((self.n_droplets, D))
        pop_pos[:, :n_feat]  = rng.uniform(self.w_min,    self.w_max,    (self.n_droplets, n_feat))
        pop_pos[:, n_feat]   = rng.uniform(self.logC_min, self.logC_max,  self.n_droplets)
        pop_pos[:, n_feat+1] = rng.uniform(self.logg_min, self.logg_max,  self.n_droplets)
        # Seed 6 droplets đầu tại vùng (C, gamma) tốt cho SVM-RBF medical tabular:
        seed_C   = [0.0, 1.0, 2.0, 0.5, 1.5, 2.5]   # log10(C): 1, 10, 100, 3.16, 31.6, 316
        seed_gam = [-2.0, -2.0, -2.0, -3.0, -3.0, -1.0]  # log10(γ): 0.01, 0.01, 0.01, 0.001
        n_seed = min(6, self.n_droplets)
        for si in range(n_seed):
            pop_pos[si, n_feat]   = seed_C[si]
            pop_pos[si, n_feat+1] = seed_gam[si]

        pop_fit = np.array([self._fitness(pop_pos[i], X, y, n_feat, cv)
                            for i in range(self.n_droplets)])

        personal_best_pos    = pop_pos.copy()
        personal_best_fit    = pop_fit.copy()
        stagnation_counter   = np.zeros(self.n_droplets, dtype=int)

        gbest_idx = np.argmin(pop_fit)
        gbest_pos = pop_pos[gbest_idx].copy()
        gbest_fit = pop_fit[gbest_idx]

        if self.verbose:
            print(f'  Init    | best={1-gbest_fit:.4f} | C={10**gbest_pos[n_feat]:.4f} | γ={10**gbest_pos[n_feat+1]:.6f}')

        # Ghi history iter 0
        self.hist_best.append(1.0 - gbest_fit)
        self.hist_mean.append(float(np.mean(1.0 - pop_fit)))
        self.hist_std.append(float(np.std(1.0 - pop_fit)))
        self.alpha_hist.append(self._alpha(0))
        self.bl_hist.append(self._beta_local(0))
        self.bg_hist.append(self._beta_global(0))
        self.levy_hist.append(self._levy_strength(0))
        self.hist_C.append(10.0 ** gbest_pos[n_feat])
        self.hist_gam.append(10.0 ** gbest_pos[n_feat + 1])

        for t in range(1, self.n_iter + 1):
            alpha    = self._alpha(t)
            beta_l   = self._beta_local(t)
            beta_g   = self._beta_global(t)
            levy_str = self._levy_strength(t)
            nb_size  = self._nb_size(t)

            # Tìm elite (top-k theo fitness thấp nhất)
            sorted_idx = np.argsort(pop_fit)
            elite_idx  = sorted_idx[:self.elite_k]
            elite_mean = pop_pos[elite_idx].mean(axis=0)

            # ── Cơ chế 3: Pre-compute pairwise distances 1 lần/iter (vectorized) ──
            # Thay vì tính O(n²) lần trong inner loop, tính 1 lần trước:
            diff = pop_pos[:, np.newaxis, :] - pop_pos[np.newaxis, :, :]   # (N,N,D)
            dist_matrix = np.sqrt((diff**2).sum(axis=2))                    # (N,N)
            np.fill_diagonal(dist_matrix, np.inf)

            for i in range(self.n_droplets):
                pos = pop_pos[i].copy()

                # ── Cơ chế 1: Raindrop Stratification ──────────
                # PSO chuẩn: thu hút bởi cả gbest VÀ personal_best
                r1 = rng.uniform(0, 1, D)
                r2 = rng.uniform(0, 1, D)
                R  = rng.uniform(-1, 1, D)
                pos = (pos
                       + alpha  * r1 * (gbest_pos           - pos)
                       + beta_l * r2 * (personal_best_pos[i] - pos)
                       + 0.05   * R)

                # ── Cơ chế 2: Global Attraction ─────────────────
                # Kéo thẳng về gbest — đúng nghĩa "global attraction"
                r3  = rng.uniform(0, 1, D)
                pos = pos + beta_g * r3 * (gbest_pos - pos)

                # ── Cơ chế 3: Neighbourhood Sharing (dùng dist_matrix đã tính) ──
                nb_idx      = np.argsort(dist_matrix[i])[:nb_size]
                nb_best_idx = nb_idx[np.argmin(pop_fit[nb_idx])]
                pos = pos + 0.1 * (pop_pos[nb_best_idx] - pos)

                # ── Cơ chế 4: Condensation (elite perturbation) ──
                if rng.random() < self.cond_pct:
                    noise = rng.normal(0, self.cond_sigma, D)
                    pos   = elite_mean + noise

                # ── Cơ chế 5: Levy Surge — chỉ khi stagnation ───
                if stagnation_counter[i] >= self.stagnation_thresh:
                    levy_step = self._levy(D, rng=rng)
                    levy_step = np.clip(levy_step, -self.surge_str, self.surge_str)
                    pos = pos + levy_str * levy_step

                pos = self._clip(pos, n_feat)
                f   = self._fitness(pos, X, y, n_feat, cv)

                # ── Greedy acceptance: chỉ chấp nhận nếu cải thiện ─
                if f < pop_fit[i]:
                    pop_pos[i] = pos
                    pop_fit[i] = f
                    stagnation_counter[i] = 0
                else:
                    stagnation_counter[i] += 1

                # Cập nhật personal best
                if f < personal_best_fit[i]:
                    personal_best_fit[i] = f
                    personal_best_pos[i] = pos.copy()

                # Cập nhật global best
                if f < gbest_fit:
                    gbest_fit = f
                    gbest_pos = pos.copy()

            # Ghi history mỗi iteration
            self.hist_best.append(1.0 - gbest_fit)
            self.hist_mean.append(float(np.mean(1.0 - pop_fit)))
            self.hist_std.append(float(np.std(1.0 - pop_fit)))
            self.alpha_hist.append(alpha)
            self.bl_hist.append(beta_l)
            self.bg_hist.append(beta_g)
            self.levy_hist.append(levy_str)
            self.hist_C.append(10.0 ** gbest_pos[n_feat])
            self.hist_gam.append(10.0 ** gbest_pos[n_feat + 1])

            if self.verbose and (t % 15 == 0 or t == 1 or t == self.n_iter):
                phase = '[EXPLORE]' if alpha > (self.alpha_max + self.alpha_min) / 2 else '[EXPLOIT]'
                print(f'  {phase} iter={t:4d}/{self.n_iter} | best={1-gbest_fit:.4f} | mean={np.mean(1-pop_fit):.4f} | C={10**gbest_pos[n_feat]:6.3f} | γ={10**gbest_pos[n_feat+1]:.6f} | α={alpha:.3f}')

        # ── Lưu kết quả cuối ──────────────────────────────────
        self.best_weights_ = gbest_pos[:n_feat]
        self.best_C_       = 10.0 ** gbest_pos[n_feat]
        self.best_gamma_   = 10.0 ** gbest_pos[n_feat + 1]
        self.best_fitness_ = 1.0 - gbest_fit
        self.convergence_  = self.hist_best   # alias cho tương thích

        return self

print('✔  DeepDOA class (fixed) định nghĩa thành công!')
print('  5 cơ chế: Raindrop(PSO chuẩn) | GlobalAttract | NeighbourShare | Condensation | LevySurge(stagnation)')


✔  DeepDOA class (fixed) định nghĩa thành công!
  5 cơ chế: Raindrop(PSO chuẩn) | GlobalAttract | NeighbourShare | Condensation | LevySurge(stagnation)


## Section 3 — Run Optimisation

In [6]:

# 3. RUN OPTIMISATION
print("  Running Deep DOA v3  (n_droplets=30, n_iter=2000, cv=5-fold) …\n")
t0 = time.time()

# ══════════════════════════════════════════════════════════════
#  THÔNG SỐ TỐI ƯU (v3) — mục tiêu ≥ 85% Acc / F1 / AUC-ROC
#  Thay đổi so với v2:
#    n_droplets : 25 → 30   (quần thể lớn hơn, đa dạng hơn)
#    n_iter     : 2000 (giữ nguyên — tốc độ tăng nhờ các tối ưu khác)
#    logC_max   : 3 → 4     (search C lên 10^4, tốt hơn cho SVM-RBF)
#    logg_min   : -3 → -4   (search gamma nhỏ hơn, tránh over-fit)
#    alpha      : 0.95→0.40 → 0.90→0.30  (khai thác mạnh hơn)
#    bl_init/end: 0.30/0.62 → 0.35/0.70  (pbest attraction mạnh hơn)
#    bg_init/end: 0.65/0.22 → 0.70/0.20  (gbest attraction mạnh hơn lúc đầu)
#    levy_init  : 0.10 → 0.15  (thoát local optima tốt hơn)
#    cond_pct   : 0.20 → 0.25  (condensation thường xuyên hơn)
#    cond_sigma : 0.15 → 0.10  (nhiễu nhỏ hơn → khai thác tinh hơn)
#    surge_every: 15 → 10   (levy surge thường xuyên hơn)
#    surge_str  : 0.65 → 0.80  (levy bước lớn hơn để thoát optima)
#    nb_init/max: 4/12 → 5/15  (xét hàng xóm nhiều hơn)
#    elite_k    : 3 → 5     (condensation học từ top-5 thay vì top-3)
#    stagnation : 20 → 12   (kích hoạt levy sớm hơn khi bị kẹt)
#  ── TỐI ƯU TỐC ĐỘ (giữ n_iter=2000) ──────────────────────
#    probability: True → False trong _fitness (nhanh hơn ~30%)
#    Neighbourhood: tính dist_matrix 1 lần/iter (O(n²)→O(n))
#    n_jobs=-1  : song song hóa cross_val_score
# ══════════════════════════════════════════════════════════════
# ══════════════════════════════════════════════════════════════
#  THÔNG SỐ v4 — mục tiêu > 85% Accuracy / F1 / AUC-ROC
#  Fix gốc rễ:
#    [FIX-1] _fitness dùng Pipeline(StandardScaler+SVM) + scoring=roc_auc
#            → C, gamma tối ưu nhất quán với hold-out evaluation
#    [FIX-2] Smart init: 6 droplets seeded tại vùng C/gamma tốt
#    [FIX-3] Tham số tinh chỉnh:
#      logC_min : -1 → 0    (C>=1, loại bỏ vùng under-regularized)
#      alpha_min: 0.30→0.20 (exploit mạnh hơn cuối vòng lặp)
#      bl_end   : 0.70→0.80 (pbest attraction mạnh hơn = nhớ kinh nghiệm)
#      bg_end   : 0.20→0.15 (giảm dần global pull ở giai đoạn exploit)
#      cond_sigma:0.10→0.08 (nhiễu tinh hơn quanh elite)
#      elite_k  : 5 → 7    (học từ top-7)
#      stagnation:12 → 10  (levy surge sớm hơn)
# ══════════════════════════════════════════════════════════════
doa = DeepDOA(
    n_droplets=30, n_iter=2000,
    w_min=0.001, w_max=5.0,
    logC_min=0.0, logC_max=4.0,
    logg_min=-4.0, logg_max=1.0,
    alpha_max=0.90, alpha_min=0.20,
    bl_init=0.35, bl_end=0.80,
    bg_init=0.70, bg_end=0.15,
    levy_init=0.15, levy_min=0.002,
    cond_pct=0.25, cond_sigma=0.08,
    surge_every=10, surge_str=0.80,
    nb_init=5, nb_max=15,
    elite_k=7, cv_folds=5,
    stagnation_thresh=10,
    verbose=True,
)
doa.optimize(X_raw, y)
elapsed = time.time() - t0

best_w   = doa.best_weights_
best_C   = doa.best_C_
best_gam = doa.best_gamma_
best_cv  = doa.best_fitness_

print(f"\n  ✔  Done in {elapsed/60:.1f} min")
print(f"  Best CV Accuracy : {best_cv:.4f}  ({best_cv*100:.2f}%)")
print(f"  Best C           : {best_C:.4f}")
print(f"  Best gamma       : {best_gam:.6f}")

  Running Deep DOA v3  (n_droplets=30, n_iter=2000, cv=5-fold) …

  Init    | best=0.8934 | C=10.0000 | γ=0.010000
  [EXPLORE] iter=   1/2000 | best=0.8941 | mean=0.8813 | C= 6.348 | γ=0.019749 | α=0.900
  [EXPLORE] iter=  15/2000 | best=0.8949 | mean=0.8947 | C= 4.184 | γ=0.013699 | α=0.895
  [EXPLORE] iter=  30/2000 | best=0.8958 | mean=0.8956 | C= 1.869 | γ=0.022823 | α=0.890
  [EXPLORE] iter=  45/2000 | best=0.8976 | mean=0.8974 | C= 1.053 | γ=0.031760 | α=0.884
  [EXPLORE] iter=  60/2000 | best=0.8976 | mean=0.8975 | C= 1.053 | γ=0.031760 | α=0.879
  [EXPLORE] iter=  75/2000 | best=0.8976 | mean=0.8975 | C= 1.053 | γ=0.031760 | α=0.874
  [EXPLORE] iter=  90/2000 | best=0.8976 | mean=0.8975 | C= 1.053 | γ=0.031760 | α=0.869
  [EXPLORE] iter= 105/2000 | best=0.8979 | mean=0.8977 | C= 1.000 | γ=0.073684 | α=0.863
  [EXPLORE] iter= 120/2000 | best=0.8979 | mean=0.8978 | C= 1.000 | γ=0.073684 | α=0.858
  [EXPLORE] iter= 135/2000 | best=0.8979 | mean=0.8978 | C= 1.000 | γ=0.073940 | α=0

## Section 4 — Exploratory Data Analysis (EDA)

In [7]:

# 4. EXPLORATORY DATA ANALYSIS VISUALISATIONS

print("\n  Generating EDA visualisations …")

from scipy import stats

fig_eda = plt.figure(figsize=(20, 14))
gs_eda  = gridspec.GridSpec(3, 3, figure=fig_eda, hspace=0.40, wspace=0.35)
colors_palette = ['#FF6B6B', '#4ECDC4']

ax_4_1 = fig_eda.add_subplot(gs_eda[0, 0])
target_counts = pd.Series(y).value_counts()
bars_target = ax_4_1.bar(['No Disease', 'Disease'],
                         [target_counts[0], target_counts[1]],
                         color=colors_palette, edgecolor='white', linewidth=1.5, width=0.5)
ax_4_1.set_title('4.1 Target Distribution', fontsize=12, fontweight='bold')
ax_4_1.set_ylabel('Count', fontsize=10)
for bar, count in zip(bars_target, [target_counts[0], target_counts[1]]):
    height = bar.get_height()
    ax_4_1.text(bar.get_x() + bar.get_width()/2., height,
                f'{int(count)}\n({count/len(y)*100:.1f}%)',
                ha='center', va='bottom', fontsize=9, fontweight='bold')

for idx, feat in enumerate(FEATURES[:3]):
    ax = fig_eda.add_subplot(gs_eda[1, idx % 3])
    feat_no_disease = X_raw[y == 0, FEATURES.index(feat)]
    feat_disease = X_raw[y == 1, FEATURES.index(feat)]
    ax.hist(feat_no_disease, bins=15, alpha=0.6, label='No Disease', color=colors_palette[0], edgecolor='white')
    ax.hist(feat_disease, bins=15, alpha=0.6, label='Disease', color=colors_palette[1], edgecolor='white')
    ax.set_title(f'4.2 {feat}', fontsize=10, fontweight='bold')
    ax.set_xlabel(feat, fontsize=9); ax.set_ylabel('Frequency', fontsize=9)
    ax.legend(fontsize=8); ax.grid(alpha=0.3, linestyle='--')

ax_4_4 = fig_eda.add_subplot(gs_eda[2, 0])
if 'sex' in FEATURES:
    sex_disease = data[data['target'] == 1]['sex'].value_counts()
    sex_no_disease = data[data['target'] == 0]['sex'].value_counts()
    x_pos = np.arange(2); width = 0.35
    ax_4_4.bar(x_pos - width/2, [sex_no_disease.get(0, 0), sex_no_disease.get(1, 0)],
               width, label='No Disease', color=colors_palette[0], edgecolor='white')
    ax_4_4.bar(x_pos + width/2, [sex_disease.get(0, 0), sex_disease.get(1, 0)],
               width, label='Disease', color=colors_palette[1], edgecolor='white')
    ax_4_4.set_title('4.4 Sex Distribution', fontsize=10, fontweight='bold')
    ax_4_4.set_ylabel('Count', fontsize=9); ax_4_4.set_xticks(x_pos)
    ax_4_4.set_xticklabels(['Female (0)', 'Male (1)'], fontsize=9)
    ax_4_4.legend(fontsize=8); ax_4_4.grid(axis='y', alpha=0.3, linestyle='--')

ax_4_4_2 = fig_eda.add_subplot(gs_eda[2, 1])
if 'cp' in FEATURES:
    cp_disease = data[data['target'] == 1]['cp'].value_counts().sort_index()
    cp_no_disease = data[data['target'] == 0]['cp'].value_counts().sort_index()
    n_cp = max(len(cp_disease), len(cp_no_disease))
    x_pos = np.arange(n_cp); width = 0.35
    ax_4_4_2.bar(x_pos - width/2, [cp_no_disease.get(i, 0) for i in range(n_cp)],
                 width, label='No Disease', color=colors_palette[0], edgecolor='white')
    ax_4_4_2.bar(x_pos + width/2, [cp_disease.get(i, 0) for i in range(n_cp)],
                 width, label='Disease', color=colors_palette[1], edgecolor='white')
    ax_4_4_2.set_title('4.4 Chest Pain Type (cp)', fontsize=10, fontweight='bold')
    ax_4_4_2.set_ylabel('Count', fontsize=9); ax_4_4_2.set_xticks(x_pos)
    ax_4_4_2.set_xticklabels([f'Type {i}' for i in range(n_cp)], fontsize=9)
    ax_4_4_2.legend(fontsize=8); ax_4_4_2.grid(axis='y', alpha=0.3, linestyle='--')

ax_4_4_3 = fig_eda.add_subplot(gs_eda[2, 2])
if 'thal' in FEATURES:
    thal_disease = data[data['target'] == 1]['thal'].value_counts().sort_index()
    thal_no_disease = data[data['target'] == 0]['thal'].value_counts().sort_index()
    n_thal = max(len(thal_disease), len(thal_no_disease))
    x_pos = np.arange(n_thal); width = 0.35
    ax_4_4_3.bar(x_pos - width/2, [thal_no_disease.get(i, 0) for i in range(n_thal)],
                 width, label='No Disease', color=colors_palette[0], edgecolor='white')
    ax_4_4_3.bar(x_pos + width/2, [thal_disease.get(i, 0) for i in range(n_thal)],
                 width, label='Disease', color=colors_palette[1], edgecolor='white')
    ax_4_4_3.set_title('4.4 Thalassemia (thal)', fontsize=10, fontweight='bold')
    ax_4_4_3.set_ylabel('Count', fontsize=9); ax_4_4_3.set_xticks(x_pos)
    ax_4_4_3.set_xticklabels([f'Type {i}' for i in range(n_thal)], fontsize=9)
    ax_4_4_3.legend(fontsize=8); ax_4_4_3.grid(axis='y', alpha=0.3, linestyle='--')

fig_eda.suptitle('EDA Visualisations — UCI Heart Disease Dataset\nTarget Distribution, Numerical Features KDE, & Categorical Features',
                fontsize=13, fontweight='bold', y=0.995)
eda_out = os.path.expanduser('~/Downloads/heart_eda_visualisations.png')
plt.savefig(eda_out, dpi=150, bbox_inches='tight', facecolor='white')
print(f"  [EDA PLOT] Saved → {eda_out}")
plt.show()


  Generating EDA visualisations …
  [EDA PLOT] Saved → /Users/minhquan/Downloads/heart_eda_visualisations.png


## Section 4.6 — Pairplot: Numerical Features

In [8]:

# 4.6 PAIRPLOT — NUMERICAL FEATURES

print("\n  Generating Pairplot visualization …")
import seaborn as sns

key_features_for_pair = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
key_features_idx = [FEATURES.index(f) for f in key_features_for_pair if f in FEATURES]
key_features_names = [f for f in key_features_for_pair if f in FEATURES]

pair_data = pd.DataFrame(X_raw[:, key_features_idx], columns=key_features_names)
pair_data['target'] = y
pair_data['target_label'] = pair_data['target'].map({0: 'No Disease', 1: 'Disease'})

fig_pair = sns.pairplot(pair_data, hue='target_label',
                        palette={'No Disease': '#FF6B6B', 'Disease': '#4ECDC4'},
                        diag_kind='kde', plot_kws={'alpha': 0.6, 's': 30},
                        corner=False, height=2.0)
fig_pair.fig.suptitle('4.6 Pairplot — Numerical Features\nRelationships between Age, Blood Pressure, Cholesterol, Heart Rate & ST Depression',
                      fontsize=12, fontweight='bold', y=0.995)
pair_out = os.path.expanduser('~/Downloads/heart_pairplot.png')
plt.savefig(pair_out, dpi=150, bbox_inches='tight', facecolor='white')
print(f"  [PAIRPLOT] Saved → {pair_out}")
plt.show()
plt.close(fig_pair.fig)


  Generating Pairplot visualization …
  [PAIRPLOT] Saved → /Users/minhquan/Downloads/heart_pairplot.png


## Section 4.7 — Regression Plots: Age vs Key Features

In [9]:
# 4.7 REGRESSION PLOTS — AGE VS KEY FEATURES
print("  Generating Regression Plots …")

regression_features = ['trestbps', 'chol', 'thalach', 'oldpeak']
reg_features_idx = [FEATURES.index(f) for f in regression_features if f in FEATURES]
reg_features_names = [f for f in regression_features if f in FEATURES]

fig_reg = plt.figure(figsize=(16, 10))
gs_reg = gridspec.GridSpec(2, 2, figure=fig_reg, hspace=0.35, wspace=0.30)
age_idx = FEATURES.index('age')

for plot_idx, (feat_name, feat_idx) in enumerate(zip(reg_features_names, reg_features_idx)):
    ax_reg = fig_reg.add_subplot(gs_reg[plot_idx // 2, plot_idx % 2])
    for target_val, label, color in [(0, 'No Disease', '#FF6B6B'), (1, 'Disease', '#4ECDC4')]:
        mask = y == target_val
        ax_reg.scatter(X_raw[mask, age_idx], X_raw[mask, feat_idx],
                      alpha=0.5, s=25, color=color, label=label, edgecolors='none')
    z_nd = np.polyfit(X_raw[y == 0, age_idx], X_raw[y == 0, feat_idx], 1)
    p_nd = np.poly1d(z_nd)
    age_range = np.linspace(X_raw[:, age_idx].min(), X_raw[:, age_idx].max(), 100)
    ax_reg.plot(age_range, p_nd(age_range), '--', color='#FF6B6B', lw=2.5, alpha=0.8, label='Trend: No Disease')
    z_d = np.polyfit(X_raw[y == 1, age_idx], X_raw[y == 1, feat_idx], 1)
    p_d = np.poly1d(z_d)
    ax_reg.plot(age_range, p_d(age_range), '--', color='#4ECDC4', lw=2.5, alpha=0.8, label='Trend: Disease')
    ax_reg.set_xlabel('Age (years)', fontsize=10, fontweight='bold')
    ax_reg.set_ylabel(feat_name.replace('_', ' ').title(), fontsize=10, fontweight='bold')
    ax_reg.set_title(f'Age vs {feat_name.upper()}', fontsize=11, fontweight='bold')
    ax_reg.legend(fontsize=9, loc='best'); ax_reg.grid(alpha=0.3, linestyle='--')

fig_reg.suptitle('4.7 Regression Plots — Age vs Key Features\nLinear Regression by Disease Status',
                fontsize=12, fontweight='bold', y=0.995)
reg_out = os.path.expanduser('~/Downloads/heart_regression_plots.png')
plt.savefig(reg_out, dpi=150, bbox_inches='tight', facecolor='white')
print(f"  [REGRESSION PLOTS] Saved → {reg_out}\n")
plt.show()

  Generating Regression Plots …
  [REGRESSION PLOTS] Saved → /Users/minhquan/Downloads/heart_regression_plots.png



## Section 4 (Final) — Hold-out Evaluation

In [10]:
# 4. FINAL HOLD-OUT EVALUATION  (SMOTE-fixed version)
# ─────────────────────────────────────────────────────────────
# Bước 1: Split TRƯỚC — test set chỉ chứa data thực, không có synthetic
# ─────────────────────────────────────────────────────────────
X_w = X_raw * best_w
X_tr_raw, X_te, y_tr_raw, y_te = train_test_split(
    X_w, y, test_size=0.20, random_state=42, stratify=y
)

# ─────────────────────────────────────────────────────────────
# Bước 2: SMOTE chỉ apply trên TRAIN set
# → Test set hoàn toàn sạch, không bị ảnh hưởng bởi synthetic samples
# ─────────────────────────────────────────────────────────────
try:
    from imblearn.over_sampling import SMOTE
    smote = SMOTE(random_state=42, k_neighbors=3)
    X_tr, y_tr = smote.fit_resample(X_tr_raw, y_tr_raw)
    print(f"  [SMOTE] Train: {X_tr_raw.shape[0]} → {X_tr.shape[0]} samples")
    unique_tr, counts_tr = np.unique(y_tr, return_counts=True)
    for u, c in zip(unique_tr, counts_tr):
        label = 'Disease' if u == 1 else 'No Disease'
        print(f"    {label:<15}: {c:4d} ({c/len(y_tr)*100:5.1f}%)")
except ImportError:
    X_tr, y_tr = X_tr_raw, y_tr_raw
    print("  ⚠  imbalanced-learn not installed. Skipping SMOTE.")

print(f"  [TEST SET]  {X_te.shape[0]} samples (data thực, không có synthetic)")
unique_te, counts_te = np.unique(y_te, return_counts=True)
for u, c in zip(unique_te, counts_te):
    label = 'Disease' if u == 1 else 'No Disease'
    print(f"    {label:<15}: {c:4d} ({c/len(y_te)*100:5.1f}%)")

# ─────────────────────────────────────────────────────────────
# Bước 3: Scale (fit trên train, transform test)
# ─────────────────────────────────────────────────────────────
sc = StandardScaler()
X_tr_s = sc.fit_transform(X_tr)
X_te_s = sc.transform(X_te)

# ─────────────────────────────────────────────────────────────
# Bước 4: Train & Evaluate
# ─────────────────────────────────────────────────────────────
svm = SVC(C=best_C, kernel='rbf', gamma=best_gam, probability=True, random_state=42)
svm.fit(X_tr_s, y_tr)
yp    = svm.predict(X_te_s)
yprob = svm.predict_proba(X_te_s)[:, 1]

acc = accuracy_score(y_te, yp)
auc = roc_auc_score(y_te, yprob)
f1  = f1_score(y_te, yp)
mcc = matthews_corrcoef(y_te, yp)
ap  = average_precision_score(y_te, yprob)

print("\n" + "═"*70)
print("  FINAL RESULTS (20% hold-out — real data only, no SMOTE leakage)")
print("═"*70)
print(f"  Accuracy  : {acc*100:.2f}%")
print(f"  AUC-ROC   : {auc:.4f}")
print(f"  F1-Score  : {f1:.4f}")
print(f"  MCC       : {mcc:.4f}")
print(f"  Avg Prec  : {ap:.4f}")
print()
print(classification_report(y_te, yp, target_names=['No Disease','Disease']))


  [SMOTE] Train: 736 → 814 samples
    No Disease     :  407 ( 50.0%)
    Disease        :  407 ( 50.0%)
  [TEST SET]  184 samples (data thực, không có synthetic)
    No Disease     :   82 ( 44.6%)
    Disease        :  102 ( 55.4%)

══════════════════════════════════════════════════════════════════════
  FINAL RESULTS (20% hold-out — real data only, no SMOTE leakage)
══════════════════════════════════════════════════════════════════════
  Accuracy  : 84.78%
  AUC-ROC   : 0.9165
  F1-Score  : 0.8667
  MCC       : 0.6913
  Avg Prec  : 0.9292

              precision    recall  f1-score   support

  No Disease       0.86      0.79      0.82        82
     Disease       0.84      0.89      0.87       102

    accuracy                           0.85       184
   macro avg       0.85      0.84      0.84       184
weighted avg       0.85      0.85      0.85       184



## Section 4B — SVM Analysis (Support Vectors, Decision Function, Kernel Comparison)

In [11]:
# 4B. PHAN TICH SUPPORT VECTOR MACHINE
print("\n  Generating SVM Analysis visualisations …")

from sklearn.decomposition import PCA

# PCA 2D for visualization
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_tr_s)
X_te_pca = pca.transform(X_te_s)

# Train SVM on PCA data for visualization
svm_pca = SVC(C=best_C, kernel='rbf', gamma=best_gam, probability=True, random_state=42)
svm_pca.fit(X_pca, y_tr)

fig_svm = plt.figure(figsize=(20, 6))
gs_svm = gridspec.GridSpec(1, 3, figure=fig_svm, hspace=0.3, wspace=0.35)

# ── Panel 1: Support Vectors visualization ────────────────────────────────
ax_sv = fig_svm.add_subplot(gs_svm[0, 0])

x_min, x_max = X_pca[:, 0].min() - 0.5, X_pca[:, 0].max() + 0.5
y_min, y_max = X_pca[:, 1].min() - 0.5, X_pca[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 100), np.linspace(y_min, y_max, 100))
Z = svm_pca.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

ax_sv.contourf(xx, yy, Z, alpha=0.2, cmap='RdYlGn')
colors_cls = ['#e74c3c', '#2ecc71']
for cls, color, label in [(0, colors_cls[0], 'No Disease'), (1, colors_cls[1], 'Disease')]:
    mask = y_tr == cls
    ax_sv.scatter(X_pca[mask, 0], X_pca[mask, 1], c=color, s=20, alpha=0.6, label=label, edgecolors='none')

sv_idx = svm_pca.support_
ax_sv.scatter(X_pca[sv_idx, 0], X_pca[sv_idx, 1], s=80, facecolors='none',
              edgecolors='blue', linewidths=1.5, label=f'Support Vectors: {len(sv_idx)}/{len(y_tr)} ({len(sv_idx)/len(y_tr)*100:.1f}%)')

ax_sv.set_xlabel('PCA Component 1', fontsize=10)
ax_sv.set_ylabel('PCA Component 2', fontsize=10)
ax_sv.set_title(f'SVM (kernel=rbf, C={best_C:.1f}) - Support Vectors', fontsize=11, fontweight='bold')
ax_sv.legend(fontsize=8, loc='upper left')

# ── Panel 2: Decision Function & Margins ─────────────────────────────────
ax_df = fig_svm.add_subplot(gs_svm[0, 1])

Z_df = svm_pca.decision_function(np.c_[xx.ravel(), yy.ravel()])
Z_df = Z_df.reshape(xx.shape)

cf = ax_df.contourf(xx, yy, Z_df, levels=20, cmap='RdBu_r', alpha=0.8)
plt.colorbar(cf, ax=ax_df)
ax_df.contour(xx, yy, Z_df, levels=[0], colors='black', linewidths=2)
ax_df.contour(xx, yy, Z_df, levels=[-1, 1], colors='black', linewidths=1, linestyles='--')

for cls, color in [(0, colors_cls[0]), (1, colors_cls[1])]:
    mask = y_tr == cls
    ax_df.scatter(X_pca[mask, 0], X_pca[mask, 1], c=color, s=15, alpha=0.5, edgecolors='none')
ax_df.scatter(X_pca[sv_idx, 0], X_pca[sv_idx, 1], s=60, facecolors='none',
              edgecolors='blue', linewidths=1.5)

ax_df.set_xlabel('PCA Component 1', fontsize=10)
ax_df.set_ylabel('PCA Component 2', fontsize=10)
ax_df.set_title('SVM - Decision Function & Margins', fontsize=11, fontweight='bold')

# ── Panel 3: Kernel Comparison (C effect) ────────────────────────────────
ax_kc = fig_svm.add_subplot(gs_svm[0, 2])

C_values = [0.01, 0.1, 1, 10, 100]
kernels = ['linear', 'rbf', 'poly']
kernel_colors = {'linear': '#e74c3c', 'rbf': '#3498db', 'poly': '#2ecc71'}

kf_comp = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
kernel_results = {k: [] for k in kernels}
for C_val in C_values:
    for kern in kernels:
        svm_k = SVC(C=C_val, kernel=kern, random_state=42)
        score = cross_val_score(svm_k, X_pca, y_tr, cv=kf_comp, scoring='accuracy').mean()
        kernel_results[kern].append(score)

x_pos = np.arange(len(C_values))
width = 0.25
for i, (kern, color) in enumerate(kernel_colors.items()):
    offset = (i - 1) * width
    ax_kc.bar(x_pos + offset, kernel_results[kern], width, label=kern.upper(),
              color=color, alpha=0.8, edgecolor='white')

ax_kc.set_xlabel('Gia tri C (regularization)', fontsize=10)
ax_kc.set_ylabel('Test Accuracy', fontsize=10)
ax_kc.set_title('SVM - Anh huong cua Kernel va C', fontsize=11, fontweight='bold')
ax_kc.set_xticks(x_pos)
ax_kc.set_xticklabels([str(c) for c in C_values], fontsize=9)
ax_kc.legend(fontsize=9)
ax_kc.set_ylim(0, 1.05)
ax_kc.grid(axis='y', alpha=0.3, linestyle='--')

fig_svm.suptitle('PHAN TICH SUPPORT VECTOR MACHINE\nSupport Vectors | Decision Function & Margins | Kernel vs C Comparison',
                fontsize=13, fontweight='bold', y=1.02)

svm_out = os.path.expanduser('~/Downloads/heart_svm_analysis.png')
plt.savefig(svm_out, dpi=150, bbox_inches='tight', facecolor='white')
print(f"  [SVM PLOT] Saved → {svm_out}")
plt.show()

print(f"\n  Support Vectors: {len(sv_idx)} / {len(y_tr)} ({len(sv_idx)/len(y_tr)*100:.1f}%)")
print(f"  Best C = {best_C:.4f} | Best gamma = {best_gam:.6f}")


  Generating SVM Analysis visualisations …
  [SVM PLOT] Saved → /Users/minhquan/Downloads/heart_svm_analysis.png

  Support Vectors: 379 / 814 (46.6%)
  Best C = 1.0000 | Best gamma = 0.073940


## Section 5 — Full Visualisation (12 panels)

In [12]:
# 5. FULL VISUALISATION  (12 panels)
print("  Rendering visualisations …")

importance      = best_w * np.std(X_raw, axis=0)
importance_norm = importance / importance.sum()
si              = np.argsort(importance_norm)

BLUE   = '#1565C0'; ORANGE = '#E65100'; GREEN  = '#2E7D32'
PURPLE = '#6A1B9A'; TEAL   = '#00695C'; RED    = '#C62828'
LBLUE  = '#42A5F5'; GOLD   = '#F57F17'

fig = plt.figure(figsize=(22, 28))
gs  = gridspec.GridSpec(4, 3, figure=fig, hspace=0.50, wspace=0.38)
iters = np.arange(len(doa.hist_best))

ax0 = fig.add_subplot(gs[0, :2])
mean_arr = np.array(doa.hist_mean); std_arr = np.array(doa.hist_std)
ax0.fill_between(iters, np.clip(mean_arr - std_arr, 0, 1), np.clip(mean_arr + std_arr, 0, 1),
                 alpha=0.12, color=ORANGE, label='Mean ± 1 std')
ax0.fill_between(iters, mean_arr, doa.hist_best, alpha=0.10, color=BLUE)
ax0.plot(iters, doa.hist_best, color=BLUE, lw=2.5, label=f'Global Best → {best_cv*100:.2f}%')
ax0.plot(iters, mean_arr, '--', color=ORANGE, lw=1.8, alpha=0.9, label='Population Mean')
ax0.axhline(0.75, color=GREEN, lw=1.4, ls=':', label='Target 75%')
ax0.axvline(len(iters)//2, color='grey', lw=1.2, ls='--', alpha=0.5, label='Phase switch (Explore→Exploit)')
ax0.set_title('DOA Convergence — Global Best & Population Mean (5-fold CV Accuracy)', fontsize=13, fontweight='bold')
ax0.set_xlabel('Iteration'); ax0.set_ylabel('CV Accuracy')
ax0.legend(fontsize=9, loc='lower right')
ax0.set_ylim(max(0.60, min(mean_arr) - 0.03), 1.01)

ax1 = fig.add_subplot(gs[0, 2])
p_iters = np.arange(1, len(doa.alpha_hist)+1)
ax1.plot(p_iters, doa.alpha_hist, color=BLUE, lw=2, label='α (evaporation)')
ax1.plot(p_iters, doa.bl_hist, color=GREEN, lw=2, label='β_local')
ax1.plot(p_iters, doa.bg_hist, color=RED, lw=2, label='β_global')
ax1.plot(p_iters, [l*5 for l in doa.levy_hist], color=GOLD, lw=1.5, ls='--', label='Lévy scale ×5')
ax1.set_title('Adaptive Parameter\nSchedules', fontsize=11, fontweight='bold')
ax1.set_xlabel('Iteration'); ax1.set_ylabel('Value')
ax1.legend(fontsize=8)

ax2 = fig.add_subplot(gs[1, 0])
ax2.semilogy(iters, doa.hist_C, color=PURPLE, lw=2, alpha=0.85)
ax2.axhline(best_C, color=RED, lw=1.5, ls='--', label=f'Best C = {best_C:.4f}')
ax2.set_title('SVM Regularization C\nEvolution', fontsize=11, fontweight='bold')
ax2.set_xlabel('Iteration'); ax2.set_ylabel('C (log scale)')
ax2.legend(fontsize=9)

ax3 = fig.add_subplot(gs[1, 1])
ax3.semilogy(iters, doa.hist_gam, color=TEAL, lw=2, alpha=0.85)
ax3.axhline(best_gam, color=RED, lw=1.5, ls='--', label=f'Best γ = {best_gam:.6f}')
ax3.set_title('SVM Kernel γ (gamma)\nEvolution', fontsize=11, fontweight='bold')
ax3.set_xlabel('Iteration'); ax3.set_ylabel('γ (log scale)')
ax3.legend(fontsize=9)

ax4 = fig.add_subplot(gs[1, 2])
sc_plot = ax4.scatter(np.log10(doa.hist_C), np.log10(doa.hist_gam),
                      c=doa.hist_best, cmap='plasma', s=20, alpha=0.55, edgecolors='none')
ax4.scatter(np.log10(best_C), np.log10(best_gam), color='red', s=220, marker='*', zorder=10,
            label=f'Best  C={best_C:.2f}  γ={best_gam:.5f}')
cbar = plt.colorbar(sc_plot, ax=ax4, pad=0.02); cbar.set_label('CV Accuracy', fontsize=9)
ax4.set_title('Hyperparameter Search\nlog₁₀(C) vs log₁₀(γ)', fontsize=11, fontweight='bold')
ax4.set_xlabel('log₁₀(C)'); ax4.set_ylabel('log₁₀(γ)')
ax4.legend(fontsize=8)

ax5 = fig.add_subplot(gs[2, 0])
col_w = [GREEN if w >= 1.0 else RED for w in best_w]
bars = ax5.barh(FEATURES, best_w, color=col_w, edgecolor='white', height=0.65, linewidth=0.7)
ax5.axvline(1.0, color='grey', lw=1.4, ls='--', alpha=0.7, label='Uniform = 1')
ax5.set_title('Optimised Feature Weights\n(green ≥ 1, red < 1)', fontsize=11, fontweight='bold')
ax5.set_xlabel('Weight'); ax5.legend(fontsize=9)
for bar, w in zip(bars, best_w):
    ax5.text(w + 0.04, bar.get_y() + bar.get_height()/2, f'{w:.3f}', va='center', fontsize=8.5)

ax6 = fig.add_subplot(gs[2, 1])
grad = plt.cm.YlOrRd(np.linspace(0.3, 0.95, n_feat))
ax6.barh(np.array(FEATURES)[si], importance_norm[si], color=grad, edgecolor='white', linewidth=0.6)
ax6.set_title('Feature Importance\n(weight × std, normalised)', fontsize=11, fontweight='bold')
ax6.set_xlabel('Relative Importance', fontsize=10)
for j, v in enumerate(importance_norm[si]):
    ax6.text(v + 0.003, j, f'{v:.3f}', va='center', fontsize=8.5)

ax7 = fig.add_subplot(gs[2, 2], polar=True)
angles = np.linspace(0, 2*np.pi, n_feat, endpoint=False).tolist()
vals = best_w.tolist()
angles_c = angles + angles[:1]; vals_c = vals + vals[:1]
ax7.plot(angles_c, vals_c, color=BLUE, lw=2.2)
ax7.fill(angles_c, vals_c, alpha=0.20, color=BLUE)
ax7.set_xticks(angles)
ax7.set_xticklabels(FEATURES, fontsize=8.5)
ax7.set_title('Feature Weight\nRadar Chart', fontsize=11, fontweight='bold', pad=18)

ax8 = fig.add_subplot(gs[3, 0])
cm_val = confusion_matrix(y_te, yp)
ConfusionMatrixDisplay(cm_val, display_labels=['No Disease','Disease']).plot(ax=ax8, colorbar=False, cmap='Blues')
ax8.set_title(f'Confusion Matrix\nAcc={acc*100:.2f}%   F1={f1:.3f}   MCC={mcc:.3f}', fontsize=11, fontweight='bold')

ax9 = fig.add_subplot(gs[3, 1])
fpr, tpr, thr = roc_curve(y_te, yprob)
ax9.fill_between(fpr, tpr, alpha=0.12, color=BLUE)
ax9.plot(fpr, tpr, color=BLUE, lw=2.5, label=f'DOA-SVM  AUC = {auc:.4f}')
ax9.plot([0,1],[0,1],'k--', lw=1, alpha=0.4, label='Random')
opt = np.argmax(tpr - fpr)
ax9.scatter(fpr[opt], tpr[opt], color=RED, s=100, zorder=6, label=f'Best thresh ≈ {thr[opt]:.2f}')
ax9.set_title('ROC Curve', fontsize=11, fontweight='bold')
ax9.set_xlabel('False Positive Rate'); ax9.set_ylabel('True Positive Rate')
ax9.legend(fontsize=9)

ax10 = fig.add_subplot(gs[3, 2])
prec, rec, _ = precision_recall_curve(y_te, yprob)
ax10.fill_between(rec, prec, alpha=0.12, color=TEAL)
ax10.plot(rec, prec, color=TEAL, lw=2.5, label=f'AP = {ap:.4f}')
ax10.axhline(y_te.mean(), color='grey', ls='--', lw=1.2, label=f'Baseline = {y_te.mean():.2f}')
ax10.set_title('Precision-Recall Curve', fontsize=11, fontweight='bold')
ax10.set_xlabel('Recall'); ax10.set_ylabel('Precision')
ax10.legend(fontsize=9)

fig.suptitle(
    'Deep DOA-SVM │ UCI Heart Disease (Cleveland + Hungarian + VA + Switzerland)\n'
    f'Accuracy={acc*100:.2f}%   AUC={auc:.4f}   F1={f1:.4f}   MCC={mcc:.4f}   AP={ap:.4f}\n'
    f'Optimised: C={best_C:.4f}  γ={best_gam:.6f}  │  n_droplets=25  n_iter=120  cv=5-fold  runtime={elapsed/60:.1f}min',
    fontsize=13, fontweight='bold', y=0.999
)

out = os.path.expanduser('~/Downloads/heart_drizzle_svm_results.png')
plt.savefig(out, dpi=150, bbox_inches='tight', facecolor='white')
print(f"  [PLOT] Saved → {out}")
plt.show()

  Rendering visualisations …
  [PLOT] Saved → /Users/minhquan/Downloads/heart_drizzle_svm_results.png


## Section 6 — Final Summary

In [13]:
# 6. FINAL SUMMARY
print("\n" + "═"*70)
print("  FINAL SUMMARY")
print("═"*70)
print(f"  CV Accuracy   : {best_cv*100:.2f}%")
print(f"  Test Accuracy : {acc*100:.2f}%")
print(f"  AUC-ROC       : {auc:.4f}")
print(f"  F1-Score      : {f1:.4f}")
print(f"  MCC           : {mcc:.4f}")
print(f"  Avg Precision : {ap:.4f}")
print(f"  Best C        : {best_C:.4f}")
print(f"  Best gamma    : {best_gam:.6f}")
print(f"  Runtime       : {elapsed/60:.1f} min")
print()
print("  Feature weights (sorted by importance):")
sorted_feat = sorted(zip(FEATURES, best_w, importance_norm), key=lambda x: -x[2])
for feat, w, imp in sorted_feat:
    bar = '█' * int(imp * 70)
    print(f"    {feat:<12}  w={w:.4f}  imp={imp:.4f}  {bar}")
print("\n  ✔  All done.")


══════════════════════════════════════════════════════════════════════
  FINAL SUMMARY
══════════════════════════════════════════════════════════════════════
  CV Accuracy   : 89.79%
  Test Accuracy : 84.78%
  AUC-ROC       : 0.9165
  F1-Score      : 0.8667
  MCC           : 0.6913
  Avg Precision : 0.9292
  Best C        : 1.0000
  Best gamma    : 0.073940
  Runtime       : 32.5 min

  Feature weights (sorted by importance):
    chol          w=0.9204  imp=0.3641  █████████████████████████
    thalach       w=2.7212  imp=0.2516  █████████████████
    trestbps      w=2.8775  imp=0.1937  █████████████
    age           w=3.5681  imp=0.1218  ████████
    oldpeak       w=3.4050  imp=0.0131  
    ca            w=4.0768  imp=0.0106  
    thal          w=1.6486  imp=0.0097  
    restecg       w=3.1617  imp=0.0092  
    slope         w=4.1373  imp=0.0084  
    cp            w=1.7819  imp=0.0060  
    fbs           w=3.4338  imp=0.0046  
    exang         w=2.0617  imp=0.0036  
    sex       

## Section 7 — Flask Web App + Ngrok

In [14]:
# 7. FLASK WEB APP + NGROK
prediction_history = []
CSV_FILE = 'heart_disease_predictions.csv'

CSV_HEADER = [
    'Thoi_gian', 'Ten_benh_nhan', 'Tuoi', 'Gioi_tinh', 'Chieu_cao_cm', 'Can_nang_kg', 'BMI',
    'Loai_dau_nguc', 'Huyet_ap_mmHg', 'Cholesterol_mgdl', 'Duong_huyet_cao',
    'Ket_qua_ECG', 'Nhip_tim_toi_da_bpm', 'Dau_nguc_van_dong',
    'ST_Depression', 'Do_doc_ST', 'So_mach_mau_chinh', 'Thalassemia',
    'Ket_qua_du_doan', 'Xac_suat_benh_tim_%', 'Xac_suat_khoe_manh_%',
    'Do_tin_cay_%', 'Model', 'Phan_loai'
]

# Create CSV with header if not exists
if not os.path.exists(CSV_FILE):
    with open(CSV_FILE, 'w', newline='', encoding='utf-8-sig') as f:
        writer = csv.writer(f)
        writer.writerow(CSV_HEADER)


def predict_heart_disease(features_list, patient_name, patient_data):
    """Predict using trained DOA-SVM model"""
    # Scale using saved scaler
    feat_arr = np.array(features_list).reshape(1, -1)
    feat_weighted = feat_arr * best_w
    feat_scaled = sc.transform(feat_weighted)
    
    pred = svm.predict(feat_scaled)[0]
    prob = svm.predict_proba(feat_scaled)[0]
    
    disease_prob = float(prob[1]) * 100
    healthy_prob = float(prob[0]) * 100
    confidence = max(disease_prob, healthy_prob)
    
    timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    
    result = {
        'timestamp': timestamp,
        'patient_name': patient_name,
        'prediction': int(pred),
        'disease_probability': disease_prob,
        'healthy_probability': healthy_prob,
        'confidence': confidence,
        'model_name': f'DOA-SVM (C={best_C:.3f}, γ={best_gam:.5f})'
    }
    
    prediction_history.append(result)
    _save_to_csv(result, patient_data)
    return result


def _save_to_csv(result, patient_data):
    try:
        height_m = patient_data.get('height', 170) / 100
        weight = patient_data.get('weight', 70)
        bmi = round(weight / (height_m * height_m), 2)
        sex_map = {1: 'Nam', 0: 'Nữ'}
        cp_map = {1: 'Typical Angina', 2: 'Atypical Angina', 3: 'Non-anginal', 4: 'Asymptomatic'}
        ecg_map = {0: 'Normal', 1: 'ST-T Abnormality', 2: 'LV Hypertrophy'}
        slope_map = {1: 'Upsloping', 2: 'Flat', 3: 'Downsloping'}
        thal_map = {3: 'Normal', 6: 'Fixed Defect', 7: 'Reversible Defect'}
        row = [
            result['timestamp'], result['patient_name'],
            patient_data.get('age', 'N/A'), sex_map.get(patient_data.get('sex', 1), 'N/A'),
            patient_data.get('height', 'N/A'), patient_data.get('weight', 'N/A'), bmi,
            cp_map.get(patient_data.get('cp', 1), 'N/A'),
            patient_data.get('trestbps', 'N/A'), patient_data.get('chol', 'N/A'),
            'Có' if patient_data.get('fbs', 0) == 1 else 'Không',
            ecg_map.get(patient_data.get('restecg', 0), 'N/A'),
            patient_data.get('thalach', 'N/A'),
            'Có' if patient_data.get('exang', 0) == 1 else 'Không',
            patient_data.get('oldpeak', 'N/A'),
            slope_map.get(patient_data.get('slope', 1), 'N/A'),
            patient_data.get('ca', 'N/A'), thal_map.get(patient_data.get('thal', 3), 'N/A'),
            'Nguy cơ bệnh tim' if result['prediction'] == 1 else 'Khỏe mạnh',
            round(result['disease_probability'], 2), round(result['healthy_probability'], 2),
            round(result['confidence'], 2), result['model_name'],
            'Có bệnh' if result['prediction'] == 1 else 'Không bệnh'
        ]
        with open(CSV_FILE, 'a', newline='', encoding='utf-8-sig') as f:
            writer = csv.writer(f)
            writer.writerow(row)
        print(f"Đã lưu CSV: {result['patient_name']}")
    except Exception as e:
        print(f"Lỗi CSV: {e}")


print('✅ Prediction functions defined')

✅ Prediction functions defined
